# Imports

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import ViTForImageClassification, AutoImageProcessor
from safetensors.torch import load_file
from torch.utils.data import DataLoader
import os
from torch.utils.data import Dataset
from PIL import Image

D:\AI\ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Model definition

In [2]:
# Definition of the joint model
class BottleneckViT(nn.Module):
    def __init__(self, model_name, num_concepts=49, num_classes=1854):
        super(BottleneckViT, self).__init__()
        self.vit = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_concepts,
            ignore_mismatched_sizes=True
        )
        self.classifier = nn.Linear(num_concepts, num_classes)

    def forward(self, pixel_values):
        concepts = self.vit(pixel_values).logits
        logits = self.classifier(concepts)
        return logits, concepts

# Initialize the model and processor

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "google/vit-large-patch16-224"
print(f"Using device: {device}")

# Initialize model architecture only.
# Weights will be loaded dynamically in the final evaluation loop.
model = BottleneckViT(model_name, num_concepts=49, num_classes=1854)
model.to(device)
model.eval()

print("Model architecture initialized successfully.")

Using device: cuda


D:\AI\ML\.venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-large-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 1024]) in the checkpoint and torch.Size([49, 1024]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([49]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model architecture initialized successfully.


# Evaluation functions

In [4]:
def evaluate_intervention(model, dataloader, plot_path, num_concepts=49, device="cuda"):
    model.eval()

    k_values = list(range(0, num_concepts + 1))
    acc_optimal = {k: [] for k in k_values}
    acc_random = {k: [] for k in k_values}

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating Interventions"):
            pixel_values = batch["pixel_values"].to(device)
            true_concepts = batch["concept_labels"].to(device)
            true_labels = batch["labels"].to(device)

            pred_concepts = model.vit(pixel_values).logits
            batch_size = pixel_values.size(0)

            for k in k_values:
                # Optimal Intervention
                corrected_concepts_opt = pred_concepts.clone()
                if k > 0:
                    errors = torch.abs(true_concepts - pred_concepts)
                    _, topk_indices = torch.topk(errors, k, dim=1)
                    corrected_concepts_opt.scatter_(1, topk_indices, true_concepts.gather(1, topk_indices))

                logits_opt = model.classifier(corrected_concepts_opt)
                preds_opt = torch.argmax(logits_opt, dim=-1)
                acc_optimal[k].extend((preds_opt == true_labels).cpu().numpy())

                # Random Intervention
                corrected_concepts_rand = pred_concepts.clone()
                if k > 0:
                    rand_indices = torch.argsort(torch.rand(batch_size, num_concepts, device=device), dim=1)[:, :k]
                    corrected_concepts_rand.scatter_(1, rand_indices, true_concepts.gather(1, rand_indices))

                logits_rand = model.classifier(corrected_concepts_rand)
                preds_rand = torch.argmax(logits_rand, dim=-1)
                acc_random[k].extend((preds_rand == true_labels).cpu().numpy())

    mean_acc_opt = [np.mean(acc_optimal[k]) for k in k_values]
    mean_acc_rand = [np.mean(acc_random[k]) for k in k_values]

    plt.figure(figsize=(10, 6))
    plt.plot(k_values, mean_acc_opt, label="Optimal Intervention", color="blue", marker="o")
    plt.plot(k_values, mean_acc_rand, label="Random Intervention", color="orange", linestyle="--", marker="x")
    plt.xlabel("Number of intervened concepts (k)")
    plt.ylabel("Accuracy")
    plt.title("Test-Time Intervention Performance vs k")
    plt.legend()
    plt.grid(True)
    plt.savefig(plot_path)
    plt.close()

    return mean_acc_opt, mean_acc_rand

def evaluate_intervention_threshold(model, dataloader, tau_values, plot_path, device="cuda"):
    model.eval()
    results = {}

    with torch.no_grad():
        for tau in tau_values:
            all_preds = []
            total_interventions = 0
            total_samples = 0

            for batch in tqdm(dataloader, desc=f"Evaluating Threshold Tau={tau:.2f}"):
                pixel_values = batch["pixel_values"].to(device)
                true_concepts = batch["concept_labels"].to(device)
                true_labels = batch["labels"].to(device)

                pred_concepts = model.vit(pixel_values).logits
                corrected_concepts = pred_concepts.clone()

                errors = torch.abs(true_concepts - pred_concepts)
                mask = errors > tau

                corrected_concepts[mask] = true_concepts[mask]

                total_interventions += mask.sum().item()
                total_samples += pixel_values.size(0)

                logits = model.classifier(corrected_concepts)
                preds = torch.argmax(logits, dim=-1)

                all_preds.extend((preds == true_labels).cpu().numpy())

            acc = np.mean(all_preds)
            avg_k = total_interventions / total_samples if total_samples > 0 else 0
            results[tau] = {"accuracy": acc, "avg_k": avg_k}
            print(f"Tau: {tau:.2f} | Avg Interventions: {avg_k:.2f} | Accuracy: {acc:.4f}")

    # Plot and save the threshold results
    taus = list(results.keys())
    accs = [results[t]["accuracy"] for t in taus]
    avg_ks = [results[t]["avg_k"] for t in taus]

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Tau (Threshold)")
    ax1.set_ylabel("Accuracy", color="tab:blue")
    ax1.plot(taus, accs, color="tab:blue", marker="o", label="Accuracy")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2 = ax1.twinx()
    ax2.set_ylabel("Avg Interventions (k)", color="tab:red")
    ax2.plot(taus, avg_ks, color="tab:red", marker="x", linestyle="--", label="Avg Interventions")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    fig.tight_layout()
    plt.title("Threshold Intervention Performance")
    plt.savefig(plot_path)
    plt.close()

    return results

# Usage

In [5]:
class ThingsDataset(Dataset):
    def __init__(self, root_dir, concepts_file, processor):
        self.root_dir = root_dir
        self.processor = processor

        self.concepts_matrix = np.loadtxt(concepts_file)

        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])

        self.image_paths = []
        self.labels = []
        self.concepts = []

        for class_idx, class_name in enumerate(self.classes):
            class_dir = os.path.join(root_dir, class_name)
            concept_vector = self.concepts_matrix[class_idx]

            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(".jpg"):
                    self.image_paths.append(os.path.join(class_dir, img_name))
                    self.labels.append(class_idx)
                    self.concepts.append(concept_vector)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        image = Image.open(img_path).convert("RGB")

        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        concept = torch.tensor(self.concepts[idx], dtype=torch.float32)

        return {
            "pixel_values": pixel_values,
            "labels": label,
            "concept_labels": concept
        }

root_directory = "object_images"
concepts_path = "spose_embedding_49d_sorted.txt"
model_name = "google/vit-large-patch16-224"
models_folder = "models"
plots_folder = "intervention_plots"

if not os.path.exists(models_folder):
    os.makedirs(models_folder)
    print(f"Created directory: {models_folder}. Please place your model files there.")

if not os.path.exists(plots_folder):
    os.makedirs(plots_folder)

processor = AutoImageProcessor.from_pretrained(model_name)

test_dataset = ThingsDataset(
    root_dir=root_directory,
    concepts_file=concepts_path,
    processor=processor
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Total images loaded: {len(test_dataset)}")
print(f"Total classes found: {len(test_dataset.classes)}")
print(f"Total batches: {len(test_dataloader)}")

safetensors_files = []
for root, dirs, files in os.walk(models_folder):
    for file in files:
        if file.endswith(".safetensors"):
            safetensors_files.append(os.path.join(root, file))

if not safetensors_files:
    print(f"No .safetensors files found in the '{models_folder}' directory or its subdirectories.")
else:
    for model_path in safetensors_files:
        # Create a unique identifier for the model based on its parent folder and filename
        parent_dir = os.path.basename(os.path.dirname(model_path))
        model_name_base = os.path.basename(model_path).replace(".safetensors", "")
        unique_model_id = f"{parent_dir}_{model_name_base}"

        topk_plot_path = os.path.join(plots_folder, f"{unique_model_id}_topk.png")
        tau_plot_path = os.path.join(plots_folder, f"{unique_model_id}_tau.png")

        run_topk = not os.path.exists(topk_plot_path)
        run_tau = not os.path.exists(tau_plot_path)

        if not run_topk and not run_tau:
            print(f"\nSkipping {model_path}: Both intervention graphs already exist.")
            continue

        print(f"\n{'='*80}")
        print(f"Loading and Evaluating Model: {model_path}")
        print(f"{'='*80}")

        state_dict = load_file(model_path)
        model.load_state_dict(state_dict)
        model.to(device)

        if run_topk:
            print(f"\n[{model_path}] Running Top-K Intervention Analysis...")
            evaluate_intervention(model, test_dataloader, plot_path=topk_plot_path, num_concepts=49, device=device)
        else:
            print(f"\n[{model_path}] Skipping Top-K Intervention, graph already exists: {topk_plot_path}")

        if run_tau:
            print(f"\n[{model_path}] Running Threshold-based Intervention Analysis...")
            tau_values = [0.1, 0.5, 1.0, 1.5, 2.0]
            evaluate_intervention_threshold(model, test_dataloader, tau_values=tau_values, plot_path=tau_plot_path, device=device)
        else:
            print(f"\n[{model_path}] Skipping Threshold Intervention, graph already exists: {tau_plot_path}")

Total images loaded: 26107
Total classes found: 1854
Total batches: 816

Loading and Evaluating Model: models\model_ind.safetensors

[models\model_ind.safetensors] Running Top-K Intervention Analysis...


Evaluating Interventions: 100%|██████████| 816/816 [1:11:09<00:00,  5.23s/it]



[models\model_ind.safetensors] Running Threshold-based Intervention Analysis...


Evaluating Threshold Tau=0.10: 100%|██████████| 816/816 [1:30:29<00:00,  6.65s/it]


Tau: 0.10 | Avg Interventions: 3.20 | Accuracy: 0.9975


...

Tau: 1.00 | Avg Interventions: 0.15 | Accuracy: 0.8866


Evaluating Threshold Tau=1.50: 100%|██████████| 816/816 [53:09<00:00,  3.91s/it]


Tau: 1.50 | Avg Interventions: 0.03 | Accuracy: 0.8844


Evaluating Threshold Tau=2.00: 100%|██████████| 816/816 [56:20<00:00,  4.14s/it]


Tau: 2.00 | Avg Interventions: 0.01 | Accuracy: 0.8841
